<a href="https://colab.research.google.com/github/everjava/sctech/blob/feat%2Fto_class/Projeto_SCTEC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import csv
import re
from datetime import datetime
import json
import logging
import traceback

class OlistPipeline:

  logging.basicConfig(
    level=logging.ERROR,
    format='Erro no arquivo: %(filename)s na linha: %(lineno)d | Mensagem: %(message)s'
  )

  def __init__(self):
    self.SEM_CATEGORIA = 'sem categoria'
    self.CLEAN_PATTERN = re.compile(r"[^\w\s]", re.UNICODE)
    self.products: list = []
    self.orders: list = []
    # Dicionário de estatísticas usado no relatório final (Tarefa 5)
    self._stats: dict = {
      "total_produtos_lidos": 0,
      "total_pedidos_lidos": 0,
      "categorias_preenchidas": 0,       # "Sem Categoria" inserido
      "dimensoes_corrigidas": 0,          # Campos físicos preenchidos com média
      "registros_descartados": 0,         # Produtos sem nenhuma dimensão
      "datas_entrega_nulas": 0,           # order_delivered_customer_date vazio
      "cancelados_sem_entrega": 0,        # Hipótese de negócio confirmada
      "nao_cancelados_sem_entrega": 0,    # Hipótese de negócio refutada
      "indisponivel": 0,                  # Pedido indisponível
      "datas_aprovacao_convertidas": 0,   # Datas reformatadas para pt-BR

      "registros_nulos_corrigidos":0,
      "total_linhas_processadas":0,
      "total_pedidos_cancelados":0,
      }


  def read_csv(self, path: str) -> list:
    print("read_csv")
    try:
      result = []
      with open(path, mode='r', encoding='utf-8') as arquivo:
        leitor_csv = csv.DictReader(arquivo)
        return list(leitor_csv)
    except FileNotFoundError:
      logging.error(f"Erro: O arquivo '{path}' não foi encontrado.")
    except Exception as e:
      logging.error(f"Ocorreu um erro: {e}")
      traceback.print_exc()

  def clean_product_category_name(self, name: str) -> str:
    # print("clean_product_category_name")
    try:
      name = name.strip().lower()
      return self.CLEAN_PATTERN.sub('', name)
    except Exception as e:
      logging.error(f"Ocorreu um erro: {e}")
      traceback.print_exc()
      return None

  def convert_data_br(self, data: str) -> str:
    # print("convert_data_br")
    try:
      data_str = datetime.strptime(data, "%Y-%m-%d %H:%M:%S")
      return data_str.strftime("%d/%m/%Y")
    except Exception as e:
      logging.error(f"Ocorreu um erro: {e}")
      traceback.print_exc()
      return None


  def calculate_products_average(self, linha):
    # print("calculate_products_average")
    try:
      total_weight = 0.0
      total_length = 0.0
      total_height = 0.0
      total_width = 0.0

      # Contadores individuais para médias precisas (caso falte algum dado no CSV)
      count_w = count_l = count_h = count_wd = 0

      # for linha in leitor_csv:
        # 1. Processa Peso
      if linha.get('product_weight_g'):
        try:
          total_weight += float(linha['product_weight_g'])
          count_w += 1
        except ValueError:
          pass  # Ignora se não for um número válido

        # 2. Processa Comprimento
        if linha.get('product_length_cm'):
          try:
            total_length += float(linha['product_length_cm'])
            count_l += 1
          except ValueError:
            pass
        # 3. Processa Altura
        if linha.get('product_height_cm'):
          try:
            total_height += float(linha['product_height_cm'])
            count_h += 1
          except ValueError:
            pass
         # 4. Processa Largura
        if linha.get('product_width_cm'):
          try:
            total_width += float(linha['product_width_cm'])
            count_wd += 1
          except ValueError:
            pass

      # Calcula as médias (usa ternário para evitar ZeroDivisionError se o CSV estiver vazio)
      media_weight = total_weight / count_w if count_w > 0 else 0
      media_length = total_length / count_l if count_l > 0 else 0
      media_height = total_height / count_h if count_h > 0 else 0
      media_width = total_width / count_wd if count_wd > 0 else 0

      return media_weight, media_length, media_height, media_width
    except Exception as e:
      logging.error(f"Ocorreu um erro: {e}")
      traceback.print_exc()
      return None, None, None, None

  def read_products(self, caminho_csv: str) -> list:

    try:
      self.products = self.read_csv(caminho_csv)
      self._stats["total_produtos_lidos"] = len(self.products)

      for produto in self.products:

        if not produto['product_category_name']:
          # ----------------------------------------------------------------
          # TAREFA 1-A: Tratar categoria ausente
          # ----------------------------------------------------------------
          produto['product_category_name'] = self.SEM_CATEGORIA
          self._stats["categorias_preenchidas"] += 1
        else:
          # ----------------------------------------------------------------
          # TAREFA 2: Tratar categoria ausente
          # ----------------------------------------------------------------
          produto['product_category_name'] = self.clean_product_category_name(produto['product_category_name'])

        # ----------------------------------------------------------------
        # TAREFA 1-B: Avaliar dimensões físicas
        # ----------------------------------------------------------------
        avg_weight_g, avg_length_cm, avg_height_cm, avg_width_cm = self.calculate_products_average(produto)
        strategies = {
          'product_weight_g': avg_weight_g,
          'product_length_cm': avg_length_cm,
          'product_height_cm': avg_height_cm,
          'product_width_cm': avg_width_cm
        }
        for field_dimension, avg_dimension in strategies.items():
          if not produto.get(field_dimension):
            produto[field_dimension] = avg_dimension
            self._stats["dimensoes_corrigidas"] += 1

      return self.products
    except Exception as e:
      logging.error(f"Ocorreu um erro: {e}")
      traceback.print_exc()
      return None

  def read_orders(self, caminho_csv: str) -> list:
    try:
      self.orders = self.read_csv(caminho_csv)
      self._stats["total_pedidos_lidos"] = len(self.orders)
      # ----------------------------------------------------------------
      # TAREFA 3: Hipótese — datas de entrega nulas ↔ pedidos cancelados?
      # ----------------------------------------------------------------
      for order in self.orders:
        order['order_purchase_timestamp'] = self.convert_data_br(order['order_purchase_timestamp'])
        order_delivered_customer_date = order.get("order_delivered_customer_date", "").strip()
        status = order.get("order_status", "").strip().lower()

        if not order_delivered_customer_date:
          order["order_delivered_customer_date"] = "N/A"
          self._stats["datas_entrega_nulas"] += 1

          if status == "canceled":
            # Hipótese CONFIRMADA para este registro:
            # sem data de entrega porque o pedido foi cancelado
            self._stats["cancelados_sem_entrega"] += 1
            order["delivery_hypothesis"] = "confirmed"

          elif status in ("shipped", "processing", "invoiced", "approved"):
            # Pedido ainda em andamento — entrega futura esperada
            order["delivery_hypothesis"] = "in_transit"
            self._stats["nao_cancelados_sem_entrega"] += 1

          elif status == "unavailable":
            # Pedido indisponível — entrega incerta
            order["delivery_hypothesis"] = "unavailable"
            self._stats["indisponivel"] += 1

          else:
            # Status desconhecido ou outro caso não mapeado
            order["delivery_hypothesis"] = "undefined"
            self._stats["nao_cancelados_sem_entrega"] += 1

        else:
          # Data de entrega presente — hipótese não aplicável
          order["delivery_hypothesis"] = "delivered"

      # ----------------------------------------------------------------
      # TAREFA 4: Reformatação de data de aprovação para padrão BR
      # ----------------------------------------------------------------
      order_approved_at = order.get("order_approved_at", "").strip()
      order["order_approved_at_br"] = self.convert_data_br(order_approved_at)

      if order["order_approved_at_br"]:
        self._stats["datas_aprovacao_convertidas"] += 1

      return self.orders


    except Exception as e:
      #TODO ADD LOGGER
      logging.error(f"Ocorreu um erro: {e}")
      return None

   # =======================================================================
    # TAREFA 5: RELATÓRIO DE STATUS
    # =======================================================================

  def report(self) -> None:
      """
      Exibe na tela um sumário estatístico do processamento (Tarefa 5).
        Todas as métricas são calculadas a partir do dicionário _stats,
      preenchido incrementalmente durante o processamento dos datasets.
      Nenhuma biblioteca externa é usada — apenas formatação nativa de strings.
      """
      # Cálculos derivados para o relatório
      total_produtos_validos = len(self.products)
      total_nulos_corrigidos = (
          self._stats["categorias_preenchidas"]
          + self._stats["dimensoes_corrigidas"]
      )
        # Verifica se a hipótese de negócio foi totalmente confirmada
      hipotese_status = (
          "✅ CONFIRMADA"
          if self._stats["nao_cancelados_sem_entrega"] == 0
          else f"⚠️  PARCIAL — {self._stats['nao_cancelados_sem_entrega']} registro(s) "
               f"sem entrega NÃO são cancelados"
      )

      # Linha separadora reutilizável
      separador = "=" * 60

      print(separador)
      print("       RELATÓRIO DE SANITIZAÇÃO — OLIST PIPELINE")
      print(separador)

      print("\n📦  PRODUTOS")
      print(f"  Total de produtos lidos    : {self._stats['total_produtos_lidos']}")
      print(f"  Registros descartados       : {self._stats['registros_descartados']}")
      print(f"  Registros válidos           : {total_produtos_validos}")
      print(f"  Categorias preenchidas      : {self._stats['categorias_preenchidas']}")
      print(f"  Dimensões corrigidas (média): {self._stats['dimensoes_corrigidas']}")

      print("\n🛒  PEDIDOS")
      print(f"  Total de pedidos lidos    : {self._stats['total_pedidos_lidos']}")
      print(f"  Datas de entrega ausentes   : {self._stats['datas_entrega_nulas']}")
      print(f"  Pedidos cancelados s/ data  : {self._stats['cancelados_sem_entrega']}")
      print(f"  Datas aprovação convertidas : {self._stats['datas_aprovacao_convertidas']}")
      print(f"  Indefinidos                 : {self._stats['indisponivel']}")

      print("\n🛒  TOTAL")
      print(f"  Total de registros lidos    : {self._stats['total_produtos_lidos'] + self._stats['total_pedidos_lidos']}")

      print("\n🔍  HIPÓTESE DE NEGÓCIO")
      print(f"  'Datas de entrega nulas = cancelados?' → {hipotese_status}")

      print("\n📊  RESUMO GERAL")
      print(f"  Total de nulos corrigidos   : {total_nulos_corrigidos}")
      print(f"  Total de cancelados         : {self._stats['cancelados_sem_entrega']}")
      print(f"  Base sanitizada?            : {'✅ SIM' if self._stats['registros_descartados'] == 0 else '⚠️  Registros descartados presentes'}")

      print(separador)



def main():
    pipeline = OlistPipeline()
    products = pipeline.read_products('sample_data/olist_products_dataset.csv')
    # print(products)
    # beautiful_json = json.dumps(products, indent=4)
    # print(beautiful_json)

    orders = pipeline.read_orders('sample_data/olist_orders_dataset.csv')
    beautiful_json = json.dumps(orders, indent=4)

    pipeline.report()
    # print(beautiful_json)

if __name__ == '__main__':
  main()






Streaming output truncated to the last 5000 lines.
product_height_cm 6.0
product_width_cm 11.0
product_weight_g 440.0
product_length_cm 21.0
product_height_cm 13.0
product_width_cm 18.0
product_weight_g 700.0
product_length_cm 30.0
product_height_cm 30.0
product_width_cm 30.0
product_weight_g 150.0
product_length_cm 20.0
product_height_cm 2.0
product_width_cm 14.0
product_weight_g 4050.0
product_length_cm 40.0
product_height_cm 12.0
product_width_cm 38.0
product_weight_g 180.0
product_length_cm 19.0
product_height_cm 13.0
product_width_cm 14.0
product_weight_g 350.0
product_length_cm 16.0
product_height_cm 4.0
product_width_cm 11.0
product_weight_g 1100.0
product_length_cm 36.0
product_height_cm 30.0
product_width_cm 27.0
product_weight_g 900.0
product_length_cm 17.0
product_height_cm 22.0
product_width_cm 17.0
product_weight_g 308.0
product_length_cm 17.0
product_height_cm 12.0
product_width_cm 13.0
product_weight_g 150.0
product_length_cm 23.0
product_height_cm 7.0
product_width_cm 1

In [15]:
import pandas as pd
import csv

with open('sample_data/olist_orders_dataset.csv', mode='r', encoding='utf-8') as arquivo:
  leitor_csv = csv.DictReader(arquivo)
  df = pd.DataFrame(leitor_csv)
  # print(df.groupby('order_status').count())
  # print(df.head())

  with open('sample_data/olist_products_dataset.csv', mode='r', encoding='utf-8') as arquivo:
    produtos = csv.DictReader(arquivo)
    df = pd.DataFrame(produtos)
    # print(df.groupby('order_status').count())
    print(df.head())
    # Substitua 'df' pelo nome do seu DataFrame
  valores_nulos = df[['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']].isnull().sum()

  # print(valores_nulos)

  colunas = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
  df_subset = df[colunas]

  # 2. Verifica cada condição:
  # - .isnull(): captura NaNs/None
  # - == 0: captura o número zero
  # - .astype(str).str.strip() == '': converte para string, limpa espaços invisíveis e checa se está vazio
  invalidos = df_subset.isnull() | (df_subset ==  '') #| (df_subset.astype(str).str.strip() == '')

  # 3. Soma a quantidade de inválidos por coluna
  total_invalidos = invalidos.sum()

  print(total_invalidos)


                         product_id  product_category_name  \
0  1e9e8ef04dbcff4541ed26657ea517e5             perfumaria   
1  3aa071139cb16b67ca9e5dea641aaa2f                  artes   
2  96bd76ec8810374ed1b65e291975717f          esporte_lazer   
3  cef67bcfe19066a932b7673e239eb23d                  bebes   
4  9dc1a7de274444849c219cff195d0b71  utilidades_domesticas   

  product_name_lenght product_description_lenght product_photos_qty  \
0                  40                        287                  1   
1                  44                        276                  1   
2                  46                        250                  1   
3                  27                        261                  1   
4                  37                        402                  4   

  product_weight_g product_length_cm product_height_cm product_width_cm  
0              225                16                10               14  
1             1000                30                

In [23]:
import csv
import re


# Substitua 'seu_arquivo.csv' pelo caminho do seu arquivo
caminho_arquivo = 'sample_data/olist_products_dataset.csv'
SEM_CATEGORIA = 'Sem Categoria'
# Compila o padrão Regex para melhor desempenho em loops
# O padrão [^\w\s] encontra tudo o que NÃO é letra, número ou espaço (incluindo pontuações e caracteres especiais)
# O caractere '_' costuma ser incluído no \w, caso queira removê-lo também, use: r"[^\w\s]|_"
CLEAN_PATTERN = re.compile(r"[^\w\s]", re.UNICODE)


try:
    with open(caminho_arquivo, mode='r', encoding='utf-8') as arquivo:
        # O DictReader transforma cada linha em um dicionário,
        # usando o cabeçalho como chave. Fica muito mais fácil de ler!
        leitor_csv = csv.DictReader(arquivo)

        # Ler todos os dados em uma lista para poder iterar várias vezes
        data = list(leitor_csv)

        print("Lendo os dados do arquivo:")
        print("-" * 30)

        # 2. Calcular a Média do comprimento do nome da categoria
        total_product_weight_g = 0
        total_product_length_cm = 0
        total_product_height_cm = 0
        total_product_width_cm = 0
        for linha in data:
            if 'product_weight_g' in linha and linha['product_weight_g']:
                total_product_weight_g += len(linha['product_weight_g'])

            if ('product_length_cm' in linha and linha['product_length_cm']):
                total_product_length_cm += len(linha['product_length_cm'])

            if ('product_height_cm' in linha and linha['product_height_cm']):
                total_product_height_cm += len(linha['product_height_cm'])

            if ('product_width_cm' in linha and linha['product_width_cm']):
                total_product_width_cm += len(linha['product_width_cm'])
        # Calcular a média

        quantidade = len(data)
        media_product_weight_g = total_product_weight_g / quantidade
        media_product_length_cm = total_product_length_cm / quantidade
        media_product_height_cm = total_product_height_cm / quantidade
        media_product_width_cm = total_product_width_cm / quantidade

        print(f"Número total de linhas: {quantidade}")
        print(f"Média do comprimento do nome da categoria: {media_product_weight_g:.2f}")
        print("-" * 30)

        for linha in data:
            if not linha['product_category_name']:
                linha['product_category_name'] = SEM_CATEGORIA
            else:
                linha['product_category_name'] = linha['product_category_name'].strip().lower()
                linha['product_category_name'] = CLEAN_PATTERN.sub('', linha['product_category_name'])

            if not linha['product_weight_g']:
                linha['product_weight_g'] = media_product_weight_g

            if not linha['product_length_cm']:
                linha['product_length_cm'] = media_product_length_cm

            if not linha['product_height_cm']:
                linha['product_height_cm'] = media_product_height_cm

            if not linha['product_width_cm']:
                linha['product_width_cm'] = media_product_width_cm


except FileNotFoundError:
    print(f"Erro: O arquivo '{caminho_arquivo}' não foi encontrado.")
except Exception as e:
    print(f"Ocorreu um erro: {e}")

Lendo os dados do arquivo:
------------------------------
Número total de linhas: 32951
Média do comprimento do nome da categoria: 3.45
------------------------------
